# Import Library

In [1]:
import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi

import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.1 MB/s eta 0:00:00


In [2]:
import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('punkt_tab')
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stop words) dalam berbagai bahasa.

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Dataset

In [3]:
    # Membuat DataFrame dari hasil scrapreview
import csv
youtube_df = pd.read_csv('/kaggle/input/datasets/danielmahulae/label-komentar-yt/Label_komentar_youtube.csv',encoding="utf-8",sep=';')
youtube_df.shape

(11022, 17)

In [4]:
youtube_df.head()

,video_id,game,comment_id,author_display_name,text,published_at,like_count,text_clean,text_casefoldingText,text_slangwords,text_title,text_stemming,text_tokenizingText,text_stopword,text_akhir,sentiment,confidence
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,08/12/2025 12:00,2415,Untuk next nya tidak akan banyak Yapping Typin...,untuk next nya tidak akan banyak yapping typin...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk lanjut nya nya tidak akan banyak ngoceh ...,"['untuk', 'lanjut', 'nya', 'nya', 'tidak', 'ak...","['tidak', 'ngoceh', 'ngetik', 'yaak', 'biar', ...",tidak ngoceh ngetik yaak biar tidak colok teri...,neutral,995
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,17/02/2026 20:48,0,Aku tau arti nya itu 😂😂,aku tau arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu,"['aku', 'tahu', 'arti', 'nya', 'itu']",['arti'],arti,neutral,9884
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",17/02/2026 20:43,1,Kebanyakan yapping lu bg ga satisfying prankNya,kebanyakan yapping lu bg ga satisfying pranknya,kebanyakan ngoceh kamu abang tidak memuaskan p...,kebanyakan ngoceh kamu abang tidak memuaskan p...,banyak ngoceh kamu abang tidak muas pranknya,"['banyak', 'ngoceh', 'kamu', 'abang', 'tidak',...","['ngoceh', 'abang', 'tidak', 'muas', 'pranknya']",ngoceh abang tidak muas pranknya,negative,9994
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,17/02/2026 19:17,0,😂😂,😂😂,😂😂,😂😂,NaN,[],[],NaN,positive,9151
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,17/02/2026 13:41,0,Palir plr,palir plr,kontol kontol,kontol kontol,kontol kontol,"['kontol', 'kontol']","['kontol', 'kontol']",kontol kontol,neutral,9965


In [5]:
youtube_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11022 entries, 0 to 11021
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   video_id              11022 non-null  object
 1   game                  11022 non-null  object
 2   comment_id            11022 non-null  object
 3   author_display_name   11022 non-null  object
 4   text                  11022 non-null  object
 5   published_at          11022 non-null  object
 6   like_count            11022 non-null  int64 
 7   text_clean            11022 non-null  object
 8   text_casefoldingText  11022 non-null  object
 9   text_slangwords       11022 non-null  object
 10  text_title            11022 non-null  object
 11  text_stemming         10760 non-null  object
 12  text_tokenizingText   11022 non-null  object
 13  text_stopword         11022 non-null  object
 14  text_akhir            10593 non-null  object
 15  sentiment             11022 non-null

In [6]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = youtube_df.drop_duplicates()

# Preprocessing

In [7]:
# Unzip results
scores= clean_df['confidence'] 
polarities = clean_df['sentiment'] 

In [8]:
print("\nDistribusi Sentimen:")
print(clean_df['sentiment'].value_counts())


Distribusi Sentimen Lexicon:
sentiment
neutral     7064
positive    2580
negative    1373
Name: count, dtype: int64


Terlihat bahwa komentar didominasi oleh komentar neutral, sehingga dilakukan weight class agar model dapat memprioritaskan label dengan jumlah komentar yang sedikit

In [9]:
clean_df

,video_id,game,comment_id,author_display_name,text,published_at,like_count,text_clean,text_casefoldingText,text_slangwords,text_title,text_stemming,text_tokenizingText,text_stopword,text_akhir,sentiment,confidence
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,08/12/2025 12:00,2415,Untuk next nya tidak akan banyak Yapping Typin...,untuk next nya tidak akan banyak yapping typin...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk lanjut nya nya tidak akan banyak ngoceh ...,"['untuk', 'lanjut', 'nya', 'nya', 'tidak', 'ak...","['tidak', 'ngoceh', 'ngetik', 'yaak', 'biar', ...",tidak ngoceh ngetik yaak biar tidak colok teri...,neutral,995
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,17/02/2026 20:48,0,Aku tau arti nya itu 😂😂,aku tau arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu,"['aku', 'tahu', 'arti', 'nya', 'itu']",['arti'],arti,neutral,9884
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",17/02/2026 20:43,1,Kebanyakan yapping lu bg ga satisfying prankNya,kebanyakan yapping lu bg ga satisfying pranknya,kebanyakan ngoceh kamu abang tidak memuaskan p...,kebanyakan ngoceh kamu abang tidak memuaskan p...,banyak ngoceh kamu abang tidak muas pranknya,"['banyak', 'ngoceh', 'kamu', 'abang', 'tidak',...","['ngoceh', 'abang', 'tidak', 'muas', 'pranknya']",ngoceh abang tidak muas pranknya,negative,9994
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,17/02/2026 19:17,0,😂😂,😂😂,😂😂,😂😂,NaN,[],[],NaN,positive,9151
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,17/02/2026 13:41,0,Palir plr,palir plr,kontol kontol,kontol kontol,kontol kontol,"['kontol', 'kontol']","['kontol', 'kontol']",kontol kontol,neutral,9965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11012,MYXRsvydCb4,The Classrooms,UgxOPXah2qikeOVJvtx4AaABAg,@soedjianimarsiman,ah ah ah ah ah,15/05/2024 19:19,0,ah ah ah ah ah 😂😅,ah ah ah ah ah 😂😅,ah ah ah ah ah 😂😅,🤣😂,ah ah ah ah ah,"['ah', 'ah', 'ah', 'ah', 'ah']",[],NaN,positive,5721
11013,MYXRsvydCb4,The Classrooms,UgzZdQkh-_g3OopP04V4AaABAg,@nelaa6877,toxic mama aku kecewa,23/04/2024 23:37,0,toxic mama aku kecewa,toxic mama aku kecewa,toxic mama aku kecewa,toxic,toxic mama aku kecewa,"['toxic', 'mama', 'aku', 'kecewa']","['toxic', 'mama', 'kecewa']",toxic,neutral,984
11014,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgwmMOdg-VR2T9vwMNR4AaABAg,@KursiKosong-w9f,Prank,02/02/2026 20:16,0,prank,prank,prank,prank,kontol kecil,"['kontol', 'kecil']",['kontol'],prank,neutral,9967
11015,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgwmMOdg-VR2T9vwMNR4AaABAg,@KursiKosong-w9f,Prank,02/02/2026 20:16,0,prank,prank,prank,kebanyakan ngetik abang jadi ketahuan,kontol kecil,"['kontol', 'kecil']",['kontol'],kebanyakan ngetik abang jadi ketahuan,neutral,9967


In [10]:
# Sentimen per game berdasarkan kategori game
polarity_per_game = pd.crosstab(clean_df['game'], clean_df['sentiment'])
polarity_game = (polarity_per_game.div(polarity_per_game.sum(axis=1), axis=0))*100
print(polarity_game.round(2))

sentiment                  negative  neutral  positive
game                                                  
I Am Fish                      2.48    77.66     19.86
Mobile Legends: Bang-Bang     35.51    57.46      7.03
The Classrooms                 3.77    61.93     34.29


berdasarkan kategori game, classrooms mendapat nilai positive tertinggi, nilai negative didominasi oleh mobile legends dan nilai neutral pada game i am fish

In [11]:
# Sentimen per game berdasarkan jenis sentimen
polarity_game_2 = (polarity_per_game.div(polarity_per_game.sum(axis=0), axis=1))*100
print(polarity_game_2.round(2))

sentiment                  negative  neutral  positive
game                                                  
I Am Fish                      4.37    26.57     18.60
Mobile Legends: Bang-Bang     80.55    25.34      8.49
The Classrooms                15.08    48.09     72.91


berdasarkan kategori jenis sentimen. sama seperti sebelumnya, classrooms mendapat nilai positive tertinggi, nilai negative didominasi oleh mobile legends dan nilai neutral pada game the clasrooms

# Data Splitting

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, precision_score
import torch

In [13]:
clean_df.head(10)

,video_id,game,comment_id,author_display_name,text,published_at,like_count,text_clean,text_casefoldingText,text_slangwords,text_title,text_stemming,text_tokenizingText,text_stopword,text_akhir,sentiment,confidence
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,08/12/2025 12:00,2415,Untuk next nya tidak akan banyak Yapping Typin...,untuk next nya tidak akan banyak yapping typin...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk lanjut nya nya tidak akan banyak ngoceh ...,"['untuk', 'lanjut', 'nya', 'nya', 'tidak', 'ak...","['tidak', 'ngoceh', 'ngetik', 'yaak', 'biar', ...",tidak ngoceh ngetik yaak biar tidak colok teri...,neutral,995
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,17/02/2026 20:48,0,Aku tau arti nya itu 😂😂,aku tau arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu,"['aku', 'tahu', 'arti', 'nya', 'itu']",['arti'],arti,neutral,9884
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",17/02/2026 20:43,1,Kebanyakan yapping lu bg ga satisfying prankNya,kebanyakan yapping lu bg ga satisfying pranknya,kebanyakan ngoceh kamu abang tidak memuaskan p...,kebanyakan ngoceh kamu abang tidak memuaskan p...,banyak ngoceh kamu abang tidak muas pranknya,"['banyak', 'ngoceh', 'kamu', 'abang', 'tidak',...","['ngoceh', 'abang', 'tidak', 'muas', 'pranknya']",ngoceh abang tidak muas pranknya,negative,9994
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,17/02/2026 19:17,0,😂😂,😂😂,😂😂,😂😂,NaN,[],[],NaN,positive,9151
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,17/02/2026 13:41,0,Palir plr,palir plr,kontol kontol,kontol kontol,kontol kontol,"['kontol', 'kontol']","['kontol', 'kontol']",kontol kontol,neutral,9965
5,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgxpV21kt9zUWVQUi394AaABAg,@mohammadkikisomantri4971,Kekuatan Orang Dalaaaaaaaaam,17/02/2026 13:14,0,Kekuatan Orang Dalaam,kekuatan orang dalaam,kekuatan orang dalam,kekuatan orang dalam,kuat orang dalam,"['kuat', 'orang', 'dalam']","['kuat', 'orang']",kuat orang,neutral,9942
6,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgwJL7hzuOfixjZD-tp4AaABAg,@FaidzhArdiyanramadhan,Nod tuh ap,17/02/2026 13:10,0,Nod tuh ap,nod tuh ap,nod itu apa,nod itu apa,nod itu apa,"['nod', 'itu', 'apa']",['nod'],nod,neutral,9974
7,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgxpxqTPWdQwzyJRFHd4AaABAg,@littlebird6995,Terlalu banyak ngetik bg.. terlalu kentara,17/02/2026 12:42,0,Terlalu banyak ngetik bg terlalu kentara,terlalu banyak ngetik bg terlalu kentara,terlalu banyak ngetik abang terlalu kentara,terlalu banyak ngetik abang terlalu kentara,terlalu banyak ngetik abang terlalu kentara,"['terlalu', 'banyak', 'ngetik', 'abang', 'terl...","['ngetik', 'abang', 'kentara']",ngetik abang kentara,negative,7787
8,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzzawrPIoShAi2z6P54AaABAg,@cecepdarwin7969,Jual akun nih. Emblem mentok semua.,17/02/2026 05:28,0,Jual akun nih Emblem mentok semua,jual akun nih emblem mentok semua,jual akun ini emblem mentok semua,jual akun ini emblem mentok semua,jual akun ini emblem mentok semua,"['jual', 'akun', 'ini', 'emblem', 'mentok', 's...","['jual', 'akun', 'emblem', 'mentok']",jual akun emblem mentok,neutral,8341
9,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgwFwHcHFbfdHSVdsDF4AaABAg,@MhmmdMaul,Kebanyakan yaping bg,16/02/2026 18:40,0,Kebanyakan yaping bg,kebanyakan yaping bg,kebanyakan ngoceh abang,kebanyakan ngoceh abang,banyak ngoceh abang,"['banyak', 'ngoceh', 'abang']","['ngoceh', 'abang']",ngoceh abang,negative,9971


In [14]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11017 entries, 0 to 11021
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   video_id              11017 non-null  object
 1   game                  11017 non-null  object
 2   comment_id            11017 non-null  object
 3   author_display_name   11017 non-null  object
 4   text                  11017 non-null  object
 5   published_at          11017 non-null  object
 6   like_count            11017 non-null  int64 
 7   text_clean            11017 non-null  object
 8   text_casefoldingText  11017 non-null  object
 9   text_slangwords       11017 non-null  object
 10  text_title            11017 non-null  object
 11  text_stemming         10755 non-null  object
 12  text_tokenizingText   11017 non-null  object
 13  text_stopword         11017 non-null  object
 14  text_akhir            10588 non-null  object
 15  sentiment             11017 non-null  obj

In [15]:
# Pisahkan data menjadi fitur (tweet) dan label (sentimen)
X = clean_df['text_title']
y = clean_df['sentiment']

# Pemodelan

## IndoBERT

In [16]:
# Install library
!pip install transformers datasets accelerate -q
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [17]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from transformers import TrainingArguments, Trainer
from datasets import Dataset
import evaluate
import transformers
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from transformers import EarlyStoppingCallback

In [18]:
# Encode Label
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_labels = len(label_encoder.classes_)
num_labels

3

In [19]:
# save mapping
label_mapping = dict(zip(label_encoder.classes_, range(num_labels)))
label_mapping

{'negative': 0, 'neutral': 1, 'positive': 2}

In [ ]:
# Load indoBERT
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [21]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",    
        truncation=True,
        max_length=256
    )

In [22]:
import torch.nn.functional as F

class FocalTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Softmax untuk dapat probabilitas
        probs = F.softmax(logits, dim=-1)
        # Ambil probabilitas untuk label yang benar
        pt = torch.gather(probs, 1, labels.unsqueeze(1)).squeeze(1)
        
        # Cross Entropy standar
        ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights.to(logits.device), reduction='none')
        
        # Focal Loss formula: (1-pt)^gamma * CE
        loss = ((1 - pt) ** self.gamma * ce_loss).mean()

        return (loss, outputs) if return_outputs else loss


In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",          # folder simpan model dan log
    eval_strategy="epoch",     # evaluasi tiap epoch
    save_strategy="epoch",           # simpan model tiap epoch
    logging_strategy="epoch",        # log tiap epoch
    learning_rate=2e-5,              # learning rate default untuk BERT
    per_device_train_batch_size=16,  # batch size saat training
    per_device_eval_batch_size=16,   # batch size saat evaluasi
    num_train_epochs=10,              # jumlah epoch
    weight_decay=0.01,               # regularisasi
    label_smoothing_factor=0.1, 
    lr_scheduler_type="cosine",      # Learning rate mengecil perlahan
    warmup_ratio=0.1,                # Stabilisasi di awal training
    load_best_model_at_end=True,     # load model terbaik otomatis
    metric_for_best_model="f1",      # metric yang dipakai untuk best model
    save_total_limit=2,              # simpan maksimal 2 model terbaik
    seed=42,                          # supaya reproducible
    greater_is_better=True,
    logging_steps=50,
    push_to_hub=False,
    report_to="none"                 # Ubah ke "wandb" jika pakai tracking
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
f1_metric = evaluate.load("f1")

# Fungsi compute_metrics untuk Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred  # unpack
    predictions = np.argmax(logits, axis=-1)  # pilih kelas dengan score tertinggi
    acc = (predictions == labels).mean()      # hitung akurasi
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]
    return {
        "accuracy": acc,
        "f1": f1
    }

In [25]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

fold_accuracies = []

In [26]:
# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded)

In [27]:
text_asli_test = clean_df.loc[X_test.index, 'text'].values

In [28]:
X_train = pd.Series(X_train).reset_index(drop=True)
y_train = pd.Series(y_train).reset_index(drop=True)

X_test = pd.Series(X_test).reset_index(drop=True)
y_test = pd.Series(y_test).reset_index(drop=True)

In [29]:
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

    print(f"\n===== FOLD {fold+1} =====")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False
    
    for layer in model.roberta.encoder.layer[:6]:
        for param in layer.parameters():
            param.requires_grad = False

    # 3. Definisi Optimizer
    optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if "classifier" not in n], "lr": 1e-5},
    {"params": [p for n, p in model.named_parameters() if "classifier" in n], "lr": 1e-4}, # Lebih besar untuk classifier
        ], weight_decay=0.01)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Lanjut tokenisasi & buat dataset HuggingFace
    train_dataset = Dataset.from_dict({
        "text": X_fold_train.tolist(),
        "label": y_fold_train.tolist()
    })
    
    val_dataset = Dataset.from_dict({
        "text": X_fold_val.tolist(),
        "label": y_fold_val.tolist()
    })

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    
    # Buat trainer & train
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        optimizers=(optimizer, None),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )
    
    trainer.train()
    metrics = trainer.evaluate()
    fold_results.append(metrics["eval_f1"])

print("Average F1:", np.mean(fold_results))

In [ ]:
full_train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})
full_train_dataset = full_train_dataset.map(tokenize_function, batched=True)
full_train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [33]:
model_final = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels
)

In [34]:
optimizer_final = torch.optim.AdamW([
    {"params": [p for n, p in model_final.named_parameters() if "classifier" not in n], "lr": 8e-6}, # Lebih kecil dikit dr sebelumnya
    {"params": [p for n, p in model_final.named_parameters() if "classifier" in n], "lr": 5e-5},
], weight_decay=0.3)

In [35]:
final_trainer = FocalTrainer(
    model=model_final,
    args=training_args,
    train_dataset=full_train_dataset,
    eval_dataset=test_dataset,  
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer_final, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

final_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.431500,0.293245,0.617060,0.628145
2,0.210483,0.138986,0.821537,0.802123
3,0.145142,0.147814,0.834241,0.823199
4,0.116788,0.142837,0.856624,0.839562
5,0.090267,0.128751,0.881428,0.860787
6,0.078390,0.133370,0.893527,0.873106
7,0.062936,0.137551,0.894737,0.877972
8,0.052953,0.167755,0.909861,0.891236
9,0.052136,0.156089,0.905021,0.884734
10,0.042232,0.161498,0.906836,0.887944


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2930, training_loss=0.12828264594484923, metrics={'train_runtime': 3423.8483, 'train_samples_per_second': 27.349, 'train_steps_per_second': 0.856, 'total_flos': 1.231897021802496e+16, 'train_loss': 0.12828264594484923, 'epoch': 10.0})

In [36]:
# Evaluation

# =========================
# TRAIN SET EVALUATION
# =========================
train_predictions = final_trainer.predict(full_train_dataset)
y_train_pred = np.argmax(train_predictions.predictions, axis=1)

print("===== TRAIN SET =====")
print(classification_report(y_train, y_train_pred, target_names=label_encoder.classes_))
print("Confusion Matrix (Train):")
print(confusion_matrix(y_train, y_train_pred))


# =========================
# TEST SET EVALUATION
# =========================
test_predictions = final_trainer.predict(test_dataset)
y_test_pred = np.argmax(test_predictions.predictions, axis=1)

print("\n===== TEST SET =====")
print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


===== TRAIN SET =====
              precision    recall  f1-score   support

    negative       0.94      0.98      0.96      1167
     neutral       0.99      0.95      0.97      6004
    positive       0.91      0.98      0.94      2193

    accuracy                           0.96      9364
   macro avg       0.95      0.97      0.96      9364
weighted avg       0.97      0.96      0.96      9364

Confusion Matrix (Train):
[[1148   17    2]
 [  69 5715  220]
 [   1   34 2158]]


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



===== TEST SET =====
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86       206
     neutral       0.95      0.91      0.93      1060
    positive       0.85      0.92      0.88       387

    accuracy                           0.91      1653
   macro avg       0.88      0.90      0.89      1653
weighted avg       0.91      0.91      0.91      1653

Confusion Matrix (Test):
[[181  22   3]
 [ 33 966  61]
 [  1  29 357]]


## Logistik Regression

In [39]:
# Bagi data menjadi data latih dan data uji
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [40]:
# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=15000, min_df=5, max_df=0.8,ngram_range=(1,3),sublinear_tf=True )   #min_df itu minimum kata(yg muncul di 17 dokumen), dan max dokumen yg memuat kata tersebut
X_tfidf = tfidf.fit_transform(X_train_lr)
X_test_tfidf = tfidf.transform(X_test_lr)

In [41]:
# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

# Menampilkan hasil ekstraksi fitur
features_df

,aa,abang,abang abang,abang ada,abang aku,abang aku kaget,abang aku kejang,abang aku lagi,abang aku mau,abang aku menonton,...,your video is,your videos,youtube,youtuber,ytta,yuk,zenmatho,zilong,zombie,zoochosis
0,0.0,0.121564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.094428,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8808,0.0,0.099936,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8809,0.0,0.055318,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8810,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8811,0.0,0.110316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
!pip install optuna

In [43]:
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import cross_val_score

In [44]:
def objective(trial):
    # Perluas range pencarian C
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    # Tambahkan penalti L1/L2 untuk mengurangi overfitting
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    
    # Solver liblinear mendukung l1 dan l2
    model = LogisticRegression(
        C=C,
        penalty=penalty,
        solver="liblinear", 
        max_iter=2000,
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"])
    )

    # Gunakan StratifiedKFold agar distribusi kelas di tiap fold konsisten
    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
    score = cross_val_score(model, X_tfidf, y_train_lr, cv=skf, scoring="f1_macro").mean()

    return score


In [45]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)

[I 2026-09-12 12:26:39,625] A new study created in memory with name: no-name-5a6fac93-517b-4ab7-a5e7-7b70aed6af71
[I 2026-09-12 12:26:39,766] Trial 0 finished with value: 0.5496495916111027 and parameters: {'C': 0.06149433861416563, 'penalty': 'l1', 'class_weight': None}. Best is trial 0 with value: 0.5496495916111027.
[I 2026-09-12 12:26:39,933] Trial 1 finished with value: 0.4584005378284286 and parameters: {'C': 0.05024609071937408, 'penalty': 'l2', 'class_weight': None}. Best is trial 0 with value: 0.5496495916111027.
[I 2026-09-12 12:26:40,146] Trial 2 finished with value: 0.8291897908604628 and parameters: {'C': 1.6595951220315763, 'penalty': 'l1', 'class_weight': 'balanced'}. Best is trial 2 with value: 0.8291897908604628.
[I 2026-09-12 12:26:40,322] Trial 3 finished with value: 0.7513796092539726 and parameters: {'C': 0.15872101263398336, 'penalty': 'l1', 'class_weight': 'balanced'}. Best is trial 2 with value: 0.8291897908604628.
[I 2026-09-12 12:26:40,559] Trial 4 finished wi

Best Params: {'C': 1.532589806893736, 'penalty': 'l1', 'class_weight': 'balanced'}


In [46]:
best_lr = LogisticRegression(
    C=study.best_params["C"],
    penalty=study.best_params["penalty"],
    solver="liblinear",
    max_iter=2000,
    class_weight=study.best_params["class_weight"]
)

best_lr.fit(X_tfidf, y_train_lr)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = best_lr.predict(X_tfidf)
y_pred_test_lr = best_lr.predict(X_test_tfidf)

print("TRAIN REPORT")
print(classification_report(y_train_lr, y_pred_train_lr))

print("TEST REPORT")
print(classification_report(y_test_lr, y_pred_test_lr))

print("Confusion Matrix:")
print(confusion_matrix(y_test_lr, y_pred_test_lr))

TRAIN REPORT
              precision    recall  f1-score   support

    negative       0.88      0.90      0.89      1098
     neutral       0.91      0.93      0.92      5651
    positive       0.86      0.78      0.82      2064

    accuracy                           0.89      8813
   macro avg       0.88      0.87      0.88      8813
weighted avg       0.89      0.89      0.89      8813

TEST REPORT
              precision    recall  f1-score   support

    negative       0.82      0.84      0.83       275
     neutral       0.88      0.91      0.89      1413
    positive       0.80      0.72      0.76       516

    accuracy                           0.86      2204
   macro avg       0.83      0.82      0.83      2204
weighted avg       0.85      0.86      0.85      2204

Confusion Matrix:
[[ 231   35    9]
 [  47 1283   83]
 [   3  140  373]]


## Multi Layer Perceptron(MLP)

In [48]:
# Bagi data menjadi data latih dan data uji
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(X, y, test_size=0.25, random_state=42,stratify=y)

In [49]:
# Ekstraksi fitur dengan TF-IDF
tfidf_nn = TfidfVectorizer(max_features=12000, min_df=2, max_df=0.8,ngram_range=(1,3) )   #min_df itu minimum kata(yg muncul di 17 dokumen), dan max dokumen yg memuat kata tersebut
X_tfidf_nn = tfidf_nn.fit_transform(X_train_nn)
X_test_tfidf_nn = tfidf_nn.transform(X_test_nn)

In [50]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)

y_train_encoded = label_encoder.transform(y_train_nn)
X_resampled, y_resampled = ros.fit_resample(X_tfidf_nn, y_train_encoded)

In [51]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64,),   # Diperkecil agar tidak menghafal
    activation='relu',
    solver='adam',
    alpha=0.1,                  # Regularisasi ditingkatkan
    learning_rate_init=0.001,
    early_stopping=True,        # Aktifkan ini!
    validation_fraction=0.15,
    n_iter_no_change=10,
    max_iter=300,
    random_state=42
)
mlp.fit(X_resampled, y_resampled)

MLPClassifier(alpha=0.1, early_stopping=True, hidden_layer_sizes=(64,),
              max_iter=300, random_state=42, validation_fraction=0.15)

In [52]:
# Ubah dulu y_train_nn yang asli (teks) menjadi angka agar sama dengan hasil prediksi
y_train_encoded = label_encoder.transform(y_train_nn)
y_test_encoded = label_encoder.transform(y_test_nn)

y_pred_train_nn = mlp.predict(X_tfidf_nn.toarray())
y_pred_test_nn = mlp.predict(X_test_tfidf_nn.toarray())

print("TRAIN REPORT")
# Gunakan target_names supaya laporannya muncul dalam teks (Positive, Negative, dll)
print(classification_report(y_train_encoded, y_pred_train_nn, target_names=label_encoder.classes_))

print("TEST REPORT")
print(classification_report(y_test_encoded, y_pred_test_nn, target_names=label_encoder.classes_))

TRAIN REPORT
              precision    recall  f1-score   support

    negative       0.97      0.99      0.98      1030
     neutral       0.99      0.94      0.96      5297
    positive       0.86      0.98      0.92      1935

    accuracy                           0.95      8262
   macro avg       0.94      0.97      0.95      8262
weighted avg       0.96      0.95      0.96      8262

TEST REPORT
              precision    recall  f1-score   support

    negative       0.81      0.82      0.81       343
     neutral       0.89      0.84      0.86      1767
    positive       0.68      0.79      0.73       645

    accuracy                           0.82      2755
   macro avg       0.79      0.81      0.80      2755
weighted avg       0.83      0.82      0.82      2755



Dari ketiga model, dapat dilihat bahwa model indobert merupakan best model untuk proyek ini

# Inference(Pengujian)

In [62]:
# Input kalimat baru dari pengguna
kalimat_baru = input("Masukkan kalimat baru: ")

# Menggunakan objek tfidf yang sudah di-fit dari pelatihan sebelumnya
X_kalimat_baru = tfidf.transform([kalimat_baru])

# Memperoleh prediksi sentimen kalimat baru
prediksi_sentimen = best_lr.predict(X_kalimat_baru)

# Menampilkan hasil prediksi
if prediksi_sentimen[0] == 'positive':
    print("Sentimen: POSITIF")
elif prediksi_sentimen[0] == 'neutral':
    print("Sentimen: NETRAL")
else:
    print("Sentimen: NEGATIF")

Masukkan kalimat baru:  ngoceh


Sentimen: NEGATIF
